# Практика. Решающие деревья и случайный лес

**По лекциям 11–12.** Сначала — разобранные примеры с описанием (демонстрации 🔍),
в конце — задачи ✏️. Всё выполняется сверху вниз; датасеты — встроенные в sklearn,
интернет не нужен.

План:
1. Дерево решений: структура, путь объекта, границы решений
2. Переобучение и критерии остановки (глубина, `min_samples_leaf`)
3. Критерии информативности: энтропия, Джини, information gain — считаем руками
4. Неустойчивость деревьев и идея усреднения (композиция)
5. Бутстрап и бэггинг
6. Случайный лес: случайные признаки (`max_features`), рост числа деревьев
7. Out-of-Bag: оценка качества без тестовой выборки
8. Важность признаков: permutation importance
9. **Задачи**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_diabetes, make_moons
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, mean_squared_error

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

def plot_boundary(model, X, y, ax=None, title="", h=0.02):
    '''Ступенчатая граница решений модели на плоскости двух признаков.'''
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=25, edgecolor="k", linewidth=0.4)
    ax.set_title(title)

print("готово к работе")


## 1. Дерево решений: как оно устроено

Дерево задаёт набор **логических правил** вида «признак $x_j \le t$?» (лекция: предикаты).
Внутренняя вершина — вопрос, лист — ответ: для классификации — самый частый класс в листе,
для регрессии — среднее ответов.

Геометрически дерево режет плоскость **прямыми, параллельными осям**: каждое правило —
одна прямая, вместе они дают ступенчатую границу. Никаких «наклонных» границ одно
дерево построить не может — это его принципиальное ограничение.


In [ ]:
# 🔍 Демонстрация 1.1. Дерево глубины 2 на «двух полумесяцах»
X, y = make_moons(n_samples=300, noise=0.25, random_state=RANDOM_STATE)

tree = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE).fit(X, y)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_tree(tree, feature_names=["x1", "x2"], class_names=["0", "1"],
          filled=True, rounded=True, fontsize=10, ax=axes[0])
plot_boundary(tree, X, y, ax=axes[1], title=f"граница решений, accuracy = {tree.score(X, y):.3f}")
plt.show()


**Как читать дерево.** Корень (верхняя вершиина) спрашивает `x1 <= 0.61` — это первый,
самый «информативный» вопрос. `gini` — неопределённость вершины (0 — чистый лист),
`samples/value` — сколько объектов попало и как они распределены по классам.
Обратите внимание: глубина 2 = не более 4 листьев = не более 4 «ступенек» границы.


In [ ]:
# 🔍 Демонстрация 1.2. Путь одного объекта по дереву
def explain_path(tree, x, feature_names):
    '''Идём от корня и печатаем вопросы, которые дерево задаёт объекту x.'''
    f = tree.tree_
    node = 0
    while f.children_left[node] != -1:            # пока не лист
        feat, thr = f.feature[node], f.threshold[node]
        answer = "да" if x[feat] <= thr else "нет"
        print(f"  {feature_names[feat]} <= {thr:.2f}?  {answer}")
        node = f.children_left[node] if x[feat] <= thr else f.children_right[node]
    proba = f.value[node][0] / f.value[node].sum()
    print(f"  ЛИСТ: класс {int(np.argmax(f.value[node]))}, распределение {proba.round(2)}")

x_new = X[0]
print("объект:", x_new.round(2), "| истинный класс:", y[0])
explain_path(tree, x_new, ["x1", "x2"])


Прогноз дерева — это просто цепочка вопросов-ответов до листа. Именно поэтому дерево
**интерпретируемо**: его можно показать заказчику или врачу как список правил.


## 2. Переобучение и критерии остановки

Из лекции: дерево можно наращивать, пока в каждом листе не останется по одному объекту —
так оно **идеально** разделит любую обучающую выборку (и ничего не запомнит про мир).
Поэтому рост дерева ограничивают: глубина (`max_depth`), минимум объектов в листе
(`min_samples_leaf`), минимум объектов для разбиения (`min_samples_split`) и другие.


In [ ]:
# 🔍 Демонстрация 2.1. Глубина: недообучение -> переобучение
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, depth in zip(axes, [1, 3, 7, None]):
    t = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(X_tr, y_tr)
    name = "без ограничения" if depth is None else f"глубина {depth}"
    plot_boundary(t, X_tr, y_tr, ax=ax,
                  title=f"{name}\ntrain {t.score(X_tr, y_tr):.3f} | test {t.score(X_te, y_te):.3f}")
plt.tight_layout(); plt.show()


**Что видно:** глубина 1 — граница из одной прямой, **недообучение**. Без ограничения —
тренировочная точность 1.0, граница состоит из «островков» под отдельные точки —
**переобучение**: на тесте хуже, чем у средней глубины. Максимальная глубина —
самый важный гиперпараметр дерева.


In [ ]:
# 🔍 Демонстрация 2.2. Регрессия: ступеньки — визитная карточка дерева
xs = np.linspace(0, 3, 150)
f_true = np.sin(2 * xs)
yr = f_true + rng.normal(0, 0.25, len(xs))
Xr = xs.reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, leaf in zip(axes, [1, 10]):
    t = DecisionTreeRegressor(min_samples_leaf=leaf, random_state=RANDOM_STATE).fit(Xr, yr)
    grid = np.linspace(0, 3, 500).reshape(-1, 1)
    ax.plot(grid, t.predict(grid), "r-", lw=2, label="прогноз дерева")
    ax.plot(xs, f_true, "g--", label="истинная функция")
    ax.scatter(xs, yr, s=12, alpha=0.5, label="данные")
    mse = mean_squared_error(np.sin(2 * grid.ravel()), t.predict(grid))
    ax.set_title(f"min_samples_leaf={leaf} | MSE ≈ {mse:.3f}"); ax.legend()
plt.show()


**Что видно:** прогноз дерева в регрессии — **константа в каждом листе**, поэтому это
ступеньки. `min_samples_leaf=1` — ступеньки под каждый шумовой выброс (переобучение);
увеличение параметра сглаживает прогноз: лист «имеет право на существование»,
только если в него попало достаточно объектов.


## 3. Энтропия, Джини и information gain — считаем руками

Как дерево выбирает вопрос? Для каждой вершины ищется разбиение, которое сильнее всего
уменьшает «хаотичность». Хаотичность (impurity) измеряют:

- **энтропия:** $H(p) = -\sum_k p_k \log_2 p_k$
- **индекс Джини:** $G(p) = 1 - \sum_k p_k^2$

Чистая вершина (один класс) — ноль; смесь 50/50 — максимум. Качество разбиения —
**information gain**: «хаотичность до минус взвешенная хаотичность после». Для регрессии
роль хаотичности играет **дисперсия** ответов в вершине.


In [ ]:
# 🔍 Демонстрация 3.1. Свои функции энтропии, Джини и information gain
def entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

def gini(y):
    p = np.bincount(y) / len(y)
    return 1 - (p ** 2).sum()

def information_gain(y, y_left, y_right, criterion=gini):
    n = len(y) + 0
    w = lambda part: len(part) / n * criterion(part)
    return criterion(y) - (w(y_left) + w(y_right))

y_node = np.array([0]*9 + [1]*5)                  # вершина: 9 класса 0 и 5 класса 1
print(f"смесь 9:5       -> энтропия {entropy(y_node):.4f} | джини {gini(y_node):.4f}")
print(f"чистый лист     -> энтропия {entropy(np.zeros(9, int)):.4f} | джини {gini(np.zeros(9, int)):.4f}")

yL, yR = np.array([0]*8 + [1]*1), np.array([0]*1 + [1]*4)  # кандидат разбиения
print(f"gain разбиения  -> {information_gain(y_node, yL, yR):.4f}")
print(f"gain бесполезного (те же пропорции) -> {information_gain(y_node, y_node[:7], y_node[7:]):.4f}")


Полезное разбиение получает заметный gain; разбиение, не изменившее пропорции классов, —
почти ноль. Именно это число дерево максимизирует в каждой вершине.


In [ ]:
# 🔍 Демонстрация 3.2. Найдём лучший порог перебором — как дерево
Xi, yi = load_iris(return_X_y=True)
Xi, yi = Xi[:, 2], yi                              # один признак: длина лепестка

gains, thresholds = [], []
for t in np.unique(Xi)[:-1]:                       # все кандидаты-пороги
    left, right = yi[Xi <= t], yi[Xi > t]
    if len(left) and len(right):
        gains.append(information_gain(yi, left, right))
        thresholds.append((t + np.unique(Xi)[np.unique(Xi) > t][0]) / 2)

best = int(np.argmax(gains))
print(f"наш лучший порог: {thresholds[best]:.3f} (gain = {gains[best]:.4f})")

sk = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE).fit(Xi.reshape(-1, 1), yi)
print(f"порог sklearn:    {sk.tree_.threshold[0]:.3f}")

plt.figure(figsize=(7, 3.2))
plt.plot(thresholds, gains, ".-")
plt.axvline(thresholds[best], color="r", ls="--", label="лучший порог")
plt.xlabel("порог по длине лепестка"); plt.ylabel("information gain"); plt.legend()
plt.show()


**Совпало!** Наш перебор порогов нашёл ровно тот порог, что и sklearn. Дерево в каждой
вершине делает ровно то, что мы только что сделали руками: перебирает пороги по всем
признакам и берёт максимум gain. Построение — **жадное**: лучший вопрос сейчас,
без оглядки на будущее (из лекции: оптимальное дерево искать перебором — слишком дорого).


In [ ]:
# 🔍 Демонстрация 3.3. Gini или entropy — есть ли разница?
Xw, yw = load_wine(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
for crit in ["gini", "entropy"]:
    scores = cross_val_score(DecisionTreeClassifier(criterion=crit, random_state=RANDOM_STATE),
                             Xw, yw, cv=cv)
    print(f"{crit:<8} accuracy на 5 фолдах: {scores.mean():.3f} ± {scores.std():.3f}")


На практике разница между критериями обычно в пределах шума. Джини считается чуть быстрее
(нет логарифма) и в sklearn — значение по умолчанию.


## 4. Неустойчивость деревьев и идея композиции

Деревья **неустойчивы** (лекция 11): уберите горсть объектов — и дерево получится совсем
другим: поменялись gain'ы — поменялось первое разбиение — поменялось всё ниже.
Плохо ли это? Для одного дерева — да. Но у этого свойства есть обратная сторона:
если обучить много разных деревьев и **усреднить** их ответы, ошибки отдельных
«нерадивых» деревьев скомпенсируются. Так устроена композиция (ансамбль).


In [ ]:
# 🔍 Демонстрация 4.1. Пять деревьев на пяти подвыборках — пять разных моделей
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6))
roots = []
for ax, seed in zip(axes, range(5)):
    idx = np.random.default_rng(seed).choice(len(X_tr), size=len(X_tr), replace=False)[:int(0.6 * len(X_tr))]
    t = DecisionTreeClassifier(max_depth=4, random_state=seed).fit(X_tr[idx], y_tr[idx])
    roots.append(X_tr[idx].mean(axis=0))
    plot_boundary(t, X_tr, y_tr, ax=ax, title=f"подвыборка {seed}: test {t.score(X_te, y_te):.3f}")
plt.tight_layout(); plt.show()


**Что видно:** все деревья неплохие, но **разные**: разные границы, разные корни, разные
ошибки. Усреднение таких «спорящих» моделей и даст более стабильный ответ.


In [ ]:
# 🔍 Демонстрация 4.2. Усредним 25 деревьев вручную (голосование большинства)
trees = [DecisionTreeClassifier(max_depth=5, random_state=s).fit(
            X_tr[np.random.default_rng(s).choice(len(X_tr), len(X_tr), replace=True)],
            y_tr[np.random.default_rng(s).choice(len(X_tr), len(X_tr), replace=True)])
         for s in range(25)]

single = np.array([t.score(X_te, y_te) for t in trees])
votes = np.stack([t.predict(X_te) for t in trees])       # (25, n_test)
majority = np.array([np.bincount(votes[:, i]).argmax() for i in range(votes.shape[1])])

print(f"медианное дерево:          {np.median(single):.3f}   <- типичное дерево")
print(f"лучшее отдельное дерево:   {single.max():.3f}   <- но какое оно, мы узнали только ПОСЛЕ теста")
print(f"голосование 25 деревьев:   {accuracy_score(y_te, majority):.3f}")


**Ключевой эффект:** голосование **лучше типичного** дерева и почти никогда не проваливается
до худшего. «Лучшее дерево» мы нашли задним числом — заранее выбрать его нельзя,
а вот голосование стабильно выдаёт качество около верхушки распределения. Каждое дерево
ошибается по-своему; чтобы ансамбль ошибся, должно «не повезти» сразу большинству.
Осталось научиться получать много разных деревьев систематически — это бэггинг
и случайный лес (лекция 12).


## 5. Бутстрап и бэггинг

**Бутстреп**: из выборки размера $n$ набираем $n$ объектов **с возвращением**. Кого-то
возьмём дважды, кого-то не возьмём вовсе. Математика: каждый объект не попадёт в выборку
с вероятностью $(1-1/n)^n \approx e^{-1} \approx 36.8\%$, то есть в бутстреп-выборке
около **63.2%** уникальных объектов.

**Бэггинг (bootstrap aggregation):** обучаем $N$ деревьев каждая на своей бутстреп-выборке
и усредняем ответы. Из лекции 12: бэггинг **не уменьшает смещение** (если все модели
систематически ошибаются — ансамбль сохранит ошибку), но **уменьшает разброс** —
а у глубоких деревьев разброс как раз огромный.


In [ ]:
# 🔍 Демонстрация 5.1. Бутстреп своими руками: доля уникальных объектов
print(f"{'n':>7} {'уникальных, %':>14} {'теория 63.2%':>13}")
for n in [10, 100, 1000, 10000]:
    idx = rng.integers(0, n, n)                     # выбор с возвращением
    print(f"{n:>7} {len(np.unique(idx)) / n:>13.1%} {'':>13}")


In [ ]:
# 🔍 Демонстрация 5.2. Одно глубокое дерево vs бэггинг
deep = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_tr, y_tr)
bag = BaggingClassifier(DecisionTreeClassifier(random_state=RANDOM_STATE),
                        n_estimators=50, random_state=RANDOM_STATE).fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
plot_boundary(deep, X_tr, y_tr, ax=axes[0], title=f"одно дерево: test {deep.score(X_te, y_te):.3f}")
plot_boundary(bag, X_tr, y_tr, ax=axes[1], title=f"бэггинг 50: test {bag.score(X_te, y_te):.3f}")
axes[2].axis("off")
axes[2].text(0.02, 0.6,
    "глубокое дерево:\n  • низкое смещение\n  • огромный разброс\n\nбэггинг усредняет 50 таких деревьев:\n  • смещение почти то же\n  • разброс падает",
    fontsize=12, va="top")
plt.tight_layout(); plt.show()


Граница бэггинга заметно глаже: «островки» отдельных деревьев взаимно уничтожились.
Но у бэггинга остаётся проблема: все деревья обучаются на **всех** признаках, поэтому
сильный признак попадает в корень почти каждого дерева — деревья получаются
**похожими**, и выигрыш от усреднения меньше возможного. Лечится случайным лесом.


## 6. Случайный лес: случайность и в объектах, и в признаках

Случайный лес = бэггинг + **случайные подпространства**: при каждом разбиении вершины
дереву доступно только $q$ случайных признаков из $d$. Рекомендации из лекции:

- классификация: $q = \sqrt{d}$
- регрессия: $q = d/3$

Признаки становятся «дефицитом»: сильный признак не всегда доступен, в корни попадают
разные признаки, деревья получаются **непохожими** — усреднение работает сильнее.


In [ ]:
# 🔍 Демонстрация 6.1. Как q влияет на качество и «похожесть» деревьев (wine, d=13)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3,
                                              random_state=RANDOM_STATE, stratify=yw)
for q, label in [(1, "q=1 (почти пни)"), (4, "q=√d ≈ 4"), (13, "q=d (бэггинг)")]:
    rf = RandomForestClassifier(n_estimators=100, max_features=q, random_state=RANDOM_STATE, n_jobs=-1)
    sc = cross_val_score(rf, Xw, yw, cv=cv)
    rf.fit(Xw_tr, yw_tr)
    roots = [est.tree_.feature[0] for est in rf.estimators_]
    uniq = len(set(roots))
    print(f"{label:<18} CV {sc.mean():.3f} | разных признаков в корнях 100 деревьев: {uniq}")


**Что видно:** при $q=d$ (бэггинг) в корнях почти одни и те же сильные признаки;
при $q=\sqrt{d}$ деревья начинают с разных признаков — разнообразие растёт, качество
обычно не хуже, а часто лучше. $q=1$ — слишком бедный выбор: деревья слабые (растёт смещение).


In [ ]:
# 🔍 Демонстрация 6.2. Рост числа деревьев: плато вместо переобучения
Xd, yd = load_diabetes(return_X_y=True)                 # регрессия, d=10 -> q=d/3
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=RANDOM_STATE)

rf = RandomForestRegressor(n_estimators=1, max_features=Xd.shape[1] // 3,
                           warm_start=True, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1)
train_err, test_err, oob_err = [], [], []
for n in range(1, 201):
    rf.n_estimators = n
    rf.fit(Xd_tr, yd_tr)
    train_err.append(mean_squared_error(yd_tr, rf.predict(Xd_tr)))
    test_err.append(mean_squared_error(yd_te, rf.predict(Xd_te)))
    oob_err.append(mean_squared_error(yd_tr, rf.oob_prediction_))

plt.figure(figsize=(8, 4))
plt.plot(train_err, label="train")
plt.plot(test_err, label="test")
plt.plot(oob_err, label="out-of-bag")
plt.xlabel("число деревьев N"); plt.ylabel("MSE"); plt.legend(); plt.show()


**Что видно (лекция, слайд «Универсальный метод»):** ошибка на train всё время падает
(деревьев всё больше — всё легче «запомнить» train), но ошибка на **test и OOB быстро
выходит на плато и не растёт**. Случайный лес **не переобучается с ростом N**: деревьев
берут «с запасом» (100–300), а настраивают другие гиперпараметры.


## 7. Out-of-Bag: оценка качества без теста

Каждое дерево леса видит только свою бутстреп-выборку — около 63% объектов. Значит,
для каждого объекта есть деревья, которые его **не видели** (out-of-bag). Прогноз для
объекта = усреднение ответов только тех деревьев, для которых он был OOB. Сравнив его
с истинным ответом, получаем честную оценку качества — **без отдельной тестовой выборки**.


In [ ]:
# 🔍 Демонстрация 7.1. OOB против test
Xc, yc = load_breast_cancer(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3,
                                              random_state=RANDOM_STATE, stratify=yc)
rf = RandomForestClassifier(n_estimators=300, oob_score=True,
                            random_state=RANDOM_STATE, n_jobs=-1).fit(Xc_tr, yc_tr)
print(f"OOB accuracy:  {rf.oob_score_:.4f}   <- посчитано на train, test не участвовал")
print(f"test accuracy: {rf.score(Xc_te, yc_te):.4f}")


OOB-оценка почти совпала с тестовой — это «бесплатная» кросс-валидация. Удобно:
подбирали гиперпараметры по OOB — тестовая выборка остаётся совершенно чистой
для финальной проверки.


## 8. Важность признаков: permutation importance

Идея из лекции: если признак важен, то **перемешивание** его значений (разрыв связи
с ответом) сильно ухудшит качество модели; если признак мусорный — почти не изменит.

Плюсы метода: работает для любой модели, считается на тесте, честно учитывает
взаимодействия. Минус: у **коррелированных** признаков важность «делится» между ними —
признак может казаться ненужным, если его работу продублирует коррелированный сосед.


In [ ]:
# 🔍 Демонстрация 8.1. Permutation importance и встроенная (MDI) на diabetes
rf_d = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(Xd_tr, yd_tr)
names = load_diabetes().feature_names

perm = permutation_importance(rf_d, Xd_te, yd_te, n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
order = np.argsort(perm.importances_mean)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].barh(np.array(names)[order], perm.importances_mean[order], xerr=perm.importances_std[order])
axes[0].set_title("permutation importance (падение MSE при перемешивании)")
axes[1].barh(names, rf_d.feature_importances_)
axes[1].set_title("встроенная важность (MDI)")
plt.tight_layout(); plt.show()

top = np.array(names)[np.argsort(perm.importances_mean)[::-1][:3]]
print("топ-3 признака:", ", ".join(top))


Оба метода выделяют одних и тех же лидеров (в diabetes это `bmi` и `s5`). Различия
в деталях: MDI считается на train и «делит» важность внутри дерева, permutation —
на тесте и честнее к взаимодействиями. Важность **не означает причинность** —
это лишь то, на что модель опиралась.


## Шпаргалка

| Что | Код |
|---|---|
| дерево-классификатор | `DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, criterion="gini"|"entropy")` |
| дерево-регрессор | `DecisionTreeRegressor(min_samples_leaf=10)` — impurity = дисперсия |
| нарисовать дерево | `plot_tree(model, feature_names=..., filled=True)` |
| бэггинг | `BaggingClassifier(DecisionTreeClassifier(), n_estimators=50)` |
| случайный лес | `RandomForestClassifier(n_estimators=300, max_features="sqrt")` — классификация; `RandomForestRegressor(..., max_features=d // 3)` — регрессия |
| OOB | `RandomForest*(oob_score=True)` → `model.oob_score_`, `model.oob_prediction_` |
| важность | `permutation_importance(model, X_test, y_test, n_repeats=20)`; встроенная — `model.feature_importances_` |
| критерии остановки | `max_depth`, `min_samples_leaf`, `min_samples_split`, `min_impurity_decrease` |

---

# 9. Задачи

Датасеты: `load_iris`, `load_wine`, `load_breast_cancer`, `load_diabetes`, `make_moons` —
все уже импортированы. Каждую задачу проверяйте запуском ячейки; там, где стоит
«✍️ Ваш ответ», — напишите вывод своими словами.


### Задача 1. Дерево для ирисов
Возьмите Iris и только два признака: длину и ширину лепестка (колонки 2 и 3). Обучите
дерево глубины 3, нарисуйте `plot_tree` и границу решений `plot_boundary`, посчитайте
accuracy на тесте (30%). Объясните путь объекта `X_te[0]` по дереву (`explain_path`).

✍️ Ваш ответ: сколько листьев у дерева и как это видно на границе?


In [ ]:
# ✏️ ВАШ КОД
...


### Задача 2. Энтропия и Джини вручную
В вершине 20 объектов: 12 класса «0» и 8 класса «1». Разбиение даёт листья
(9 «0», 1 «1») и (3 «0», 7 «1»). Посчитайте **вручную на бумаге** энтропию и Джини
корня, затем information gain (по Джини). Проверьте функциями `entropy`, `gini`,
`information_gain` из раздела 3.

✍️ Ваш ответ: чему равен gain? «Полезное» ли это разбиение?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 3. Свой поиск лучшего разбиения
Возьмите wine, признаки `alcohol` (0) и `flavanoids` (6). Напишите функцию
`best_split(X, y, j)`, которая перебором порогов находит разбиение с максимальным
gain (по Джини). Примените её к обоим признакам: какой признак и порог выберет
дерево в **корне**? Сравните с `DecisionTreeClassifier(max_depth=1)` — выведите
`tree_.feature[0]` и `tree_.threshold[0]`.

✍️ Ваш ответ: совпал ли ваш корень со sklearn?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 4. Подбор глубины по кросс-валидации
Для wine постройте график CV-accuracy (`StratifiedKFold(5)`) дерева в зависимости от
`max_depth` от 1 до 15. Найдите лучшую глубину, обучите на всём train и оцените на test.

✍️ Ваш ответ: где недообучение, где переобучение и почему у кривой такая форма?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 5. Регрессия: сглаживание ступенек
Сгенерируйте `xs = linspace(0, 3, 200)`, `y = sin(2x) + N(0, 0.3)`. Обучите
`DecisionTreeRegressor` с `min_samples_leaf` = 1, 5, 20. Нарисуйте три прогноза
(как в демо 2.2) и посчитайте MSE на плотной сетке против истинного синуса.

✍️ Ваш ответ: как параметр влияет на ступеньки и ошибку?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 6. Нестабильность важностей
Обучите 10 деревьев (`max_depth=5`) на 10 случайных 60%-подвыборках breast_cancer.
Соберите их `feature_importances_` в таблицу и постройте `boxplot` для 5 самых важных
в среднем признаков.

✍️ Ваш ответ: насколько «прыгает» важность одного и того же признака между деревьями?
Что это говорит об устойчивости дерева (лекция 11)?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 7. Свой бэггинг
Напишите свой бэггинг на moons (train/test как в демо): 51 глубокое дерево, каждое —
на своей бутстреп-выборке (как в демо 4.2). Ответ ансамбля — голосование большинством.
Сравните accuracy: лучшее дерево / медианное дерево / ваш бэггинг / `BaggingClassifier`.
Нарисуйте границу вашего бэггинга.


In [ ]:
# ✏️ ВАШ КОД
...


### Задача 8 ★. Теорема Кондорсе: почему голосование работает
Смоделируйте «жюри» из 25 независимых деревьев, каждое ошибается с вероятностью 0.4
(отвечает правильно с p=0.6). Сымитируйте 10 000 задач: сколько раз жюри в целом
ошибётся (большинство ответило неверно)? Сравните с теорией
$P(\text{ошибка жюри}) = \sum_{k<13} C_{25}^{k} p^k (1-p)^{25-k}$.

✍️ Ваш ответ: при каком p жюри становится хуже одного эксперта? Проверьте кодом при p=0.4.


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 9. Настройка q = max_features
Для breast_cancer сравните по 5-fold CV случайный лес с `max_features` = 1, 2, `sqrt`,
`None` (все признаки = бэггинг), `n_estimators=200`. Постройте таблицу/график
CV-accuracy ± std.

✍️ Ваш ответ: где деревья слишком слабые (смещение), а где слишком похожие (разброс)?
Что выбрать?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 10. OOB как замена кросс-валидации
На breast_cancer постройте на **одном** train-сплите кривые: OOB-accuracy и test-accuracy
случайного леса при числе деревьев 25, 50, 100, 200, 400 (`oob_score=True`,
`warm_start=True`). Постройте обе кривые на одном графике.

✍️ Ваш ответ: насколько OOB близок к test? Зачем тогда держать отдельный test?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 11. Permutation importance своими руками
Напишите свою функцию `my_permutation_importance(model, X, y, repeats=10)`: для каждого
признака — `repeats` раз перемешать столбец и записать падение accuracy. Примените к
случайному лесу на breast_cancer (test). Сравните bar-графиком с
`sklearn.inspection.permutation_importance` и `model.feature_importances_`.

✍️ Ваш ответ: какие признаки в топе? Что значит отрицательная важность?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### Задача 12. Итоговое сравнение
Сгенерируйте `make_moons(n_samples=500, noise=0.35)`. Рядом на одном рисунке — границы
решений: одно настроенное дерево (глубину подберите сами), бэггинг и случайный лес.
Подписи — test-accuracy каждого.

✍️ Ваш ответ: чем границы принципиально отличаются и почему лес глаже?


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

### ⭐ Бонус. Коррелированные признаки обманывают важность
Добавьте к breast_cancer копию признака `worst radius` под именем `radius_clone`
(идеальная корреляция). Обучите лес и выведите permutation importance обоих.
Что произошло с важностью и почему (подсказка — лекция 12, «случайные подпространства»
и обсуждение корреляции)?

✍️ Ваш ответ: ...


In [ ]:
# ✏️ ВАШ КОД
...


✍️ Ваш ответ: ...

---

**Итоги практики:**
- дерево — набор правил «$x_j \le t$», жадно строится по gain (Джини/энтропия/дисперсия);
  легко читается, но неустойчиво и склонно к переобучению — его ограничивают;
- бэггинг усредняет деревья на бутстреп-выборках: разброс падает, смещение — нет;
- случайный лес добавляет случайный выбор $q$ признаков на каждое разбиение
  ($\sqrt{d}$ / $d/3$): деревья получаются непохожими, усреднение работает сильнее;
- лес не переобучается ростом $N$; качество можно оценивать по OOB без теста;
- permutation importance показывает, на что опиралась модель (но осторожно с корреляциями).

Дальше по курсу — градиентный бустинг: ансамбль, который борется уже со **смещением**.
